# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
I chose Logistic Regression as the first machine-learning model because the lane requires ranking content opportunities using an observed binary outcome. Logistic Regression provides an interpretable probability score that can be used to rank pages.

It is a simple starting point that allows us to determine whether learned signals improve on the transparent Week-4 rule before introducing a more complex model.

In [41]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

# Reproducibility
RANDOM_STATE = 42

# DuckDB connection
con = duckdb.connect()

print("Setup complete.")
print("DuckDB version:", duckdb.__version__)
print("Pandas version:", pd.__version__)

Setup complete.
DuckDB version: 1.3.2
Pandas version: 2.2.2


In [42]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [43]:
con.execute("DROP SECRET IF EXISTS hf_secret")

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face access configured.")

Hugging Face access configured.


## 2. Split design

I will use a client-grouped train/test split using `client_hash_id` as the grouping variable.

This is appropriate because the warehouse contains multiple content pages for the same client. A random row-level split could place pages from the same client in both training and testing, which could make the evaluation look better than performance on unseen clients.

The `client_hash_id` will only be used to create the split and will not be used as a model feature.

I will hold out 20% of clients for testing and use a fixed random seed of 42 so that the experiment is reproducible.

The model will be evaluated only on the held-out clients.

### 2A.Build the modeling dataset

In [44]:
# Section 2 — Build feature data
# April + May are used as the historical feature window

feature_data = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_90d,

        MAX(report_date) AS last_report_date

    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/data_0.parquet'
    ])

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature rows:", len(feature_data))
print("Columns:", feature_data.columns.tolist())
print(feature_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 389153
Columns: ['client_hash_id', 'content_hash_id', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'last_report_date']
            client_hash_id           content_hash_id  impressions_90d  \
0  client_62f4a7e64f5e0096  content_76c1f31e2b38f054            526.0   
1  client_62f4a7e64f5e0096  content_ffc5ab4b34aab1f8           1075.0   
2  client_62f4a7e64f5e0096  content_9739856fc83dc1ca           1181.0   
3  client_62f4a7e64f5e0096  content_d47ba5533f9c8573              0.0   
4  client_62f4a7e64f5e0096  content_50266f97d6233542             20.0   

   clicks_90d  avg_position_90d last_report_date  
0         1.0         31.283270       2026-05-31  
1         1.0         19.440930       2026-05-31  
2         0.0         21.817951       2026-05-31  
3         0.0               NaN       2026-05-31  
4         0.0         16.350000       2026-05-31  


In [45]:
# Section 2 — June outcome window
# June is kept separate from the feature window to avoid leakage

outcome_data = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_june,
        SUM(gsc_clicks) AS clicks_june,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_june

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/data_0.parquet'
    )

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Outcome rows:", len(outcome_data))
print("Clients:", outcome_data["client_hash_id"].nunique())
print(outcome_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Outcome rows: 409205
Clients: 65
            client_hash_id           content_hash_id  impressions_june  \
0  client_3ffa76342f366962  content_cde79a1a7432ce40               0.0   
1  client_3ffa76342f366962  content_902b2d9b3d8a19a2               0.0   
2  client_3ffa76342f366962  content_e70ae5bf6ab35b59               2.0   
3  client_3ffa76342f366962  content_1a2c2230fb24d7fb               0.0   
4  client_3ffa76342f366962  content_d6cacc9a22770146               0.0   

   clicks_june  avg_position_june  
0          0.0                NaN  
1          0.0                NaN  
2          0.0                8.5  
3          0.0                NaN  
4          0.0                NaN  


In [46]:
model_df = feature_data.merge(
    outcome_data,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Model rows:", len(model_df))
print(model_df.head())

Model rows: 389032
            client_hash_id           content_hash_id  impressions_90d  \
0  client_62f4a7e64f5e0096  content_76c1f31e2b38f054            526.0   
1  client_62f4a7e64f5e0096  content_ffc5ab4b34aab1f8           1075.0   
2  client_62f4a7e64f5e0096  content_9739856fc83dc1ca           1181.0   
3  client_62f4a7e64f5e0096  content_d47ba5533f9c8573              0.0   
4  client_62f4a7e64f5e0096  content_50266f97d6233542             20.0   

   clicks_90d  avg_position_90d last_report_date  impressions_june  \
0         1.0         31.283270       2026-05-31             278.0   
1         1.0         19.440930       2026-05-31            1138.0   
2         0.0         21.817951       2026-05-31             198.0   
3         0.0               NaN       2026-05-31               0.0   
4         0.0         16.350000       2026-05-31               2.0   

   clicks_june  avg_position_june  
0          1.0          47.485612  
1          1.0          32.559754  
2          0.

### Create future-looking June decline label

In [47]:
model_df["impressions_change_pct"] = (
    (model_df["impressions_june"] - model_df["impressions_90d"])
    / model_df["impressions_90d"].replace(0, np.nan)
) * 100

model_df["is_declining_label"] = (
    model_df["impressions_change_pct"] <= -30
).astype(int)

print(model_df["is_declining_label"].value_counts())
print("\nDeclining rate:",
      model_df["is_declining_label"].mean())

is_declining_label
1    215206
0    173826
Name: count, dtype: int64

Declining rate: 0.5531832856937219


###  Filter historical impressions >= 100

In [48]:
model_df = model_df[
    model_df["impressions_90d"] >= 100
].copy()

### final target

In [49]:
DECLINE_THRESHOLD = -70

model_df["is_declining_label"] = (
    model_df["impressions_change_pct"] <= DECLINE_THRESHOLD
).astype(int)

print("Final model rows:", len(model_df))
print("\nLabel distribution:")
print(model_df["is_declining_label"].value_counts())
print(
    "\nDeclining rate:",
    f"{model_df['is_declining_label'].mean():.1%}"
)

Final model rows: 136830

Label distribution:
is_declining_label
1    79318
0    57512
Name: count, dtype: int64

Declining rate: 58.0%


### 2B. Grouped split

In [50]:

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Client overlap:", len(overlap))

print("Train declining rate:",
      f"{train_df['is_declining_label'].mean():.1%}")

print("Test declining rate:",
      f"{test_df['is_declining_label'].mean():.1%}")

Train rows: 91231
Test rows: 45599
Train clients: 42
Test clients: 11
Client overlap: 0
Train declining rate: 57.7%
Test declining rate: 58.5%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [51]:
print(model_df.columns.tolist())

['client_hash_id', 'content_hash_id', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'last_report_date', 'impressions_june', 'clicks_june', 'avg_position_june', 'impressions_change_pct', 'is_declining_label']


### 3A.Preparing the model features

In [52]:
MODEL_FEATURES = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d"
]

# Create historical CTR from the feature window
for frame in [train_df, test_df]:
    frame["ctr_90d"] = np.where(
        frame["impressions_90d"] > 0,
        frame["clicks_90d"] / frame["impressions_90d"],
        0
    )

MODEL_FEATURES.append("ctr_90d")

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df["is_declining_label"]

X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df["is_declining_label"]

print("Features:", MODEL_FEATURES)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Features: ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'ctr_90d']
X_train: (91231, 4)
X_test: (45599, 4)
y_train: (91231,)
y_test: (45599,)


### 3B. Training

In [53]:

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

### 3C. Generatinng the model ranking score.

In [54]:
test_df["model_score"] = model.predict_proba(X_test)[:, 1]

top10_model = (
    test_df[
        [
            "content_hash_id",
            "client_hash_id",
            "is_declining_label",
            "model_score"
        ]
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
)

print("Top 10 Logistic Regression rankings:")
print(top10_model.to_string(index=False))

Top 10 Logistic Regression rankings:
         content_hash_id          client_hash_id  is_declining_label  model_score
content_af6dd069f6d917eb client_62f4a7e64f5e0096                   1     0.724612
content_5d84ff5db281fd29 client_62f4a7e64f5e0096                   1     0.723896
content_0ef15cf0a68603a9 client_e547b89c05043229                   1     0.723633
content_d94ca5b6391b12b3 client_23a62021009f63c4                   1     0.723399
content_f50c8f6f4a44509d client_62f4a7e64f5e0096                   1     0.722991
content_59d506f7c7c07e2d client_e547b89c05043229                   1     0.722903
content_1b24a9e3ad8e8150 client_62f4a7e64f5e0096                   0     0.722646
content_342f7253e24f27cc client_62f4a7e64f5e0096                   1     0.722301
content_c9d9b29e33136b7a client_e547b89c05043229                   0     0.722220
content_eb2530425515410c client_400c21c81c8b46ef                   1     0.722189


In [55]:
print(model_df.columns.tolist())

['client_hash_id', 'content_hash_id', 'impressions_90d', 'clicks_90d', 'avg_position_90d', 'last_report_date', 'impressions_june', 'clicks_june', 'avg_position_june', 'impressions_change_pct', 'is_declining_label']


### 3D. retrieve content update dates

In [56]:
content_metadata = con.execute("""
    SELECT
        content_hash_id,
        content_updated_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df()

print("Metadata rows:", len(content_metadata))
print(content_metadata.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Metadata rows: 519606
            content_hash_id content_updated_date
0  content_004de9653278b5a4           2026-07-01
1  content_00dc5efae381b2ab           2026-07-01
2  content_01410f2556c327ac           2026-07-01
3  content_019f27f634053ca7           2026-06-15
4  content_01efa71faea45dcc           2026-06-01


In [57]:
test_df = test_df.merge(
    content_metadata,
    on="content_hash_id",
    how="left"
)

print("Test rows:", len(test_df))
print(
    "Missing update dates:",
    test_df["content_updated_date"].isna().sum()
)

Test rows: 45599
Missing update dates: 0


### 3D. Calculating the Week-4 baseline on test data

In [58]:
from sklearn.preprocessing import MinMaxScaler
test_df["days_since_last_update"] = (
    pd.Timestamp("2026-06-30")
    - pd.to_datetime(test_df["content_updated_date"])
).dt.days

# Normalize the three Week-4 signals
baseline_features = [
    "impressions_90d",
    "avg_position_90d",
    "days_since_last_update"
]

scaler = MinMaxScaler()

test_df[
    ["imp_norm", "pos_norm", "stale_norm"]
] = scaler.fit_transform(
    test_df[baseline_features].fillna(0)
)

# Week-4 rule-based score
test_df["baseline_score"] = (
      0.50 * test_df["imp_norm"]
    + 0.30 * test_df["pos_norm"]
    + 0.20 * test_df["stale_norm"]
)

print("Baseline rows:", len(test_df))
print(
    test_df[
        [
            "content_hash_id",
            "baseline_score",
            "is_declining_label"
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
    .to_string(index=False)
)

Baseline rows: 45599
         content_hash_id  baseline_score  is_declining_label
content_eadb33b5df496f4a        0.524254                   0
content_fc2ed4186a2754eb        0.399645                   1
content_ca54b759e3e84e92        0.341280                   1
content_4b4a0b01cc54b0af        0.341147                   1
content_cb65cb0398607ec6        0.322096                   1
content_f6ef1c6de22b048e        0.306805                   1
content_0609c8c6cc7f35dc        0.301856                   0
content_a9322b74ca7cb1bb        0.298595                   1
content_d4028bc3689b4eba        0.297945                   1
content_c60628276389acbb        0.294115                   0


### 3E.  Compare model vs Week-4 baseline


In [59]:

def precision_at_k(df, score_column, k):
    top_k = df.sort_values(
        score_column,
        ascending=False
    ).head(k)

    return top_k["is_declining_label"].mean()


k_values = [10, 20, 50, 100]

comparison_rows = []

for name, score_column in [
    ("Week-4 Baseline", "baseline_score"),
    ("Logistic Regression", "model_score")
]:
    row = {"Model": name}

    for k in k_values:
        row[f"Precision@{k}"] = precision_at_k(
            test_df,
            score_column,
            k
        )

    comparison_rows.append(row)

comparison_table = pd.DataFrame(comparison_rows)

print(
    comparison_table.to_string(index=False)
)

              Model  Precision@10  Precision@20  Precision@50  Precision@100
    Week-4 Baseline           0.7          0.65          0.66           0.64
Logistic Regression           0.8          0.70          0.68           0.66


## 4. Errors and interpretation

Using a 0.5 threshold for error inspection, Logistic Regression produced **15,630 false positives**, **3,059 false negatives**, and **26,910 correct classifications** on the held-out test set.

The model also made high-confidence errors: the highest false positive had a score of **0.723**, while the lowest false negative had a score of **0.001**. This shows that the model is useful for ranking but is not perfectly predictive.

The model uses historical impressions, clicks, average position, and CTR. These are plausible performance signals, but the learned relationships are associations, not causal effects.

### Leakage and validation check

The model uses **April + May data as features** and **June as the future outcome window**. The target is:

`is_declining_label = impressions_change_pct <= -70%`

June outcome fields were not used as model features. The client ID was used only for grouped splitting.

The split contained **42 training clients and 11 test clients**, with **0 client overlap**.

### Model vs Week-4 baseline

| Model | P@10 | P@20 | P@50 | P@100 |
| --- | ---: | ---: | ---: | ---: |
| Week-4 Baseline | 0.70 | 0.65 | 0.66 | 0.64 |
| Logistic Regression | **0.80** | **0.70** | **0.68** | **0.66** |

Logistic Regression performed better than the baseline at every tested K. The largest improvement was at **Precision@10 (+10 percentage points)**.

Overall, Logistic Regression provides a modest but consistent improvement and can be used as a **decision-support ranking signal** rather than an automatic refresh decision.

### 4A. Error analysis

In [60]:
error_df = test_df.copy()

error_df["predicted"] = (
    error_df["model_score"] >= 0.5
).astype(int)

error_df["error_type"] = np.where(
    (error_df["is_declining_label"] == 0) &
    (error_df["predicted"] == 1),
    "False Positive",
    np.where(
        (error_df["is_declining_label"] == 1) &
        (error_df["predicted"] == 0),
        "False Negative",
        "Correct"
    )
)

print(error_df["error_type"].value_counts())

print("\nTop false positives:")
print(
    error_df[
        error_df["error_type"] == "False Positive"
    ][
        ["content_hash_id", "model_score", "is_declining_label"]
    ]
    .sort_values("model_score", ascending=False)
    .head(5)
    .to_string(index=False)
)

print("\nTop false negatives:")
print(
    error_df[
        error_df["error_type"] == "False Negative"
    ][
        ["content_hash_id", "model_score", "is_declining_label"]
    ]
    .sort_values("model_score")
    .head(5)
    .to_string(index=False)
)

error_type
Correct           26910
False Positive    15630
False Negative     3059
Name: count, dtype: int64

Top false positives:
         content_hash_id  model_score  is_declining_label
content_1b24a9e3ad8e8150     0.722646                   0
content_c9d9b29e33136b7a     0.722220                   0
content_0fb40ab2f405c4dc     0.721940                   0
content_df250077469f614e     0.721722                   0
content_1e98c7b61c32f90f     0.721649                   0

Top false negatives:
         content_hash_id  model_score  is_declining_label
content_52f3c0dbd8df2319     0.000928                   1
content_7d87c69eaba32c8c     0.008184                   1
content_0e03de7680314cd5     0.008774                   1
content_e8a52cf3d5988c07     0.009145                   1
content_e681a39a6d29f0b2     0.015161                   1


### 4B. Logistic Regression feature interpretation

The largest absolute coefficient was for **ctr_90d (-0.669)**, followed by **avg_position_90d (-0.192)**, **impressions_90d (-0.105)**, and **clicks_90d (-0.070)**.

These coefficients show the directional associations learned by the model. Because the numerical features were standardized, larger absolute coefficients indicate stronger influence on the model's predicted probability. These are associations rather than causal effects.

In [61]:
coefficients = model.named_steps["classifier"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": coefficients
})

feature_importance["absolute_coefficient"] = (
    feature_importance["coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print(
    feature_importance[
        ["feature", "coefficient"]
    ].to_string(index=False)
)

         feature  coefficient
         ctr_90d    -0.668889
avg_position_90d    -0.191908
 impressions_90d    -0.104866
      clicks_90d    -0.069859


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.